# 02 — Training & Evaluation

Melatih 14 konfigurasi (3 granularity × 2 student × 2 seed, plus answer-only per
student) dan mengevaluasi masing-masing pada 300 soal test yang sama. Ditambah
baseline pre-distillation: kedua student diuji tanpa training sama sekali.

**Prasyarat**
1. Accelerator: **GPU T4 x2** — hanya satu yang dipakai, lihat cell Config
2. Settings → **Internet: ON**
3. Add Data → dataset keluaran `01_data_generation`

Tiap run melatih, mengevaluasi, lalu membebaskan memori, dan menulis satu baris ke
`results.csv`. Loop melewati run yang sudah tercatat, jadi 14 run dapat dipecah
lintas beberapa sesi.

In [1]:
!pip install -q unsloth

## Config

Semua dikunci identik lintas run — ini variabel kontrol PRD §4.3, bukan ruang tuning.
Satu-satunya yang berubah antar run: varian granularity, ukuran student, seed.

In [2]:
import os
# Kaggle memberi T4 x2. HF Trainer auto-DataParallel -> effective batch diam-diam
# jadi 32, bukan 16, dan step/epoch turun separuh. Kunci ke satu GPU.
# HARUS sebelum torch di-import.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import re, gc, json, time, random
from pathlib import Path
import torch, numpy as np, pandas as pd

MODELS   = ["unsloth/Qwen3-0.6B", "unsloth/Qwen3-1.7B"]   # 0.6B dulu (PRD §9: lebih cepat)
VARIANTS = ["G1", "G2", "G3"]
SEEDS    = [3407, 42]
MAX_SEQ  = 1024          # diverifikasi di notebook 01: G3 max 780 token, 0% overflow
MAX_NEW  = 512           # headroom generasi G3 (~340 token rationale)
N_TEST   = 300

# PRD §4.3 — dikunci
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0
LR, EPOCHS, BS, GRAD_ACC = 2e-4, 3, 2, 8      # effective batch 16
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

ROOT = Path("/kaggle/working")
ADAPTERS, RESULTS = ROOT/"adapters", ROOT/"results"
for d in (ADAPTERS, RESULTS): d.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS/"results.csv"

import glob
cands = sorted(glob.glob("/kaggle/input/datasets/radityaadhirajasa/cot-paperish-porto/data/processed"))
assert cands, "Dataset belum di-attach. Add Data -> dataset 'CoT Paperish Porto'."
DATA = Path(cands[0])
if len(cands) > 1:
    print("PERINGATAN: >1 sumber data. Pakai:", DATA, "| diabaikan:", cands[1:])
need = [f"train_{v}.jsonl" for v in ["G1","G2","G3","AO"]] + ["test_300.jsonl"]
missing = [f for f in need if not (DATA/f).exists()]
assert not missing, f"file hilang di {DATA}: {missing}"
print("data:", DATA)

# PRD §4.4: T4 (Turing, CC 7.5) tidak punya bf16 native. torch.cuda.is_bf16_supported()
# mengembalikan True di T4 karena menghitung jalur emulasi -> JANGAN dipakai untuk
# memilih presisi. Compute capability >= 8 (Ampere) adalah gerbang yang benar.
CC = torch.cuda.get_device_capability()
BF16 = CC[0] >= 8
print(f"GPU : {torch.cuda.get_device_name(0)} (CC {CC[0]}.{CC[1]})")
print(f"presisi: {'bf16' if BF16 else 'fp16'}"
      f"   [is_bf16_supported() bilang {torch.cuda.is_bf16_supported()} — diabaikan]")

# gagal keras kalau CUDA_VISIBLE_DEVICES tidak berlaku (mis. kernel tidak fresh)
assert torch.cuda.device_count() == 1, (
    f"terlihat {torch.cuda.device_count()} GPU — effective batch akan jadi "
    f"{BS*GRAD_ACC*torch.cuda.device_count()}, bukan {BS*GRAD_ACC}. Restart kernel.")

data: /kaggle/input/datasets/radityaadhirajasa/cot-paperish-porto/data/processed
GPU : Tesla T4 (CC 7.5)
presisi: fp16   [is_bf16_supported() bilang True — diabaikan]


## Data + format training

Format adalah **variabel kontrol paling halus** di eksperimen ini. Satu fungsi
membangun teks untuk keempat varian; yang berbeda hanya isi `rationale`.
Jawaban selalu diakhiri `#### <angka>` supaya parser evaluasi seragam.

In [3]:
def load_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

test_rows = load_jsonl(DATA/"test_300.jsonl")[:N_TEST]
train_by_variant = {v: load_jsonl(DATA/f"train_{v}.jsonl") for v in VARIANTS + ["AO"]}

N = {v: len(r) for v, r in train_by_variant.items()}
print(f"train: {N}\ntest : {len(test_rows)}")
# kontrol PRD: N identik lintas varian, kalau tidak yang terukur efek ukuran data
assert len(set(N.values())) == 1, f"N berbeda antar varian: {N}"
assert len(test_rows) == N_TEST, f"test {len(test_rows)}, harusnya {N_TEST}"
assert [r["question"] for r in train_by_variant["G1"]] == \
       [r["question"] for r in train_by_variant["AO"]], "urutan soal berbeda antar varian"
print(f"\nN train = {next(iter(N.values()))} per varian")

INSTR = "Solve the math problem. End your reply with '#### ' followed by the final number.\n\n"

def target_text(row):
    """Satu-satunya perbedaan antar varian ada di rationale."""
    r = row.get("rationale", "").strip()
    return f"{r}\n#### {row['answer']}" if r else f"#### {row['answer']}"

def build_prompt(tokenizer, question, shots=()):
    msgs = []
    for s in shots:
        msgs += [{"role": "user", "content": INSTR + s["question"]},
                 {"role": "assistant", "content": target_text(s)}]
    msgs.append({"role": "user", "content": INSTR + question})
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def build_train_text(tokenizer, row):
    msgs = [{"role": "user", "content": INSTR + row["question"]},
            {"role": "assistant", "content": target_text(row)}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)

train: {'G1': 1455, 'G2': 1455, 'G3': 1455, 'AO': 1455}
test : 300

N train = 1455 per varian  <- cek angka ini cocok dengan versi dataset yang kamu maksud


### Parser jawaban

PRD §9 menandai parsing sebagai risiko: akurasi terukur salah kalau parser lemah.
Diuji di sini dengan kasus yang benar-benar muncul di output model.

In [4]:
def extract_answer(text):
    """Ambil angka setelah '####'. Fallback: angka terakhir di teks."""
    m = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", text)
    if not m:
        nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", text)
        if not nums: return None
        m_val = nums[-1]
    else:
        m_val = m.group(1)
    try: v = float(m_val.replace(",", "").rstrip("."))
    except ValueError: return None
    return int(v) if v == int(v) else round(v, 4)

_cases = [
    ("48 + 24 = 72\n#### 72", 72),
    ("#### 1,200", 1200),
    ("The answer is 18.\n#### 18\n<|im_end|>", 18),
    ("blah #### 3.5 blah", 3.5),
    ("no hash marks, ends with 42", 42),        # fallback
    ("#### -7", -7),
    ("tidak ada angka sama sekali", None),
]
for t, e in _cases:
    got = extract_answer(t)
    assert got == e, f"{t!r} -> {got!r}, harusnya {e!r}"
print("parser jawaban ok")

parser jawaban ok


### Verifikasi format

Cek tiga hal yang kalau salah akan merusak seluruh eksperimen secara diam-diam:
blok `<think>` bocor, format berbeda antar varian, dan target tidak diakhiri `####`.

In [5]:
from unsloth import FastLanguageModel

_, _tok = FastLanguageModel.from_pretrained(
    MODELS[0], max_seq_length=MAX_SEQ, load_in_4bit=True, full_finetuning=False)

for v in ["G1", "G3", "AO"]:
    t = build_train_text(_tok, train_by_variant[v][0])
    print(f"\n{'='*22} {v} {'='*22}\n{t}")

# gate format
#
# Qwen3 SELALU memancarkan pasangan <think></think> saat enable_thinking=False —
# itu penanda mode-non-thinking, bukan kebocoran, dan prompt inference juga
# memuatnya. Yang berbahaya adalah blok itu BERISI penalaran, karena jadi CoT
# tak terkontrol yang mencemari sumbu granularity. Jadi cek isinya, bukan ada/tidaknya.
def think_content(text):
    return [m.group(1).strip() for m in re.finditer(r"<think>(.*?)</think>", text, re.S)]

for v in VARIANTS + ["AO"]:
    for row in train_by_variant[v][:50]:
        t = build_train_text(_tok, row)
        leak = [c for c in think_content(t) if c]
        assert not leak, f"{v}: blok think berisi penalaran: {leak[0][:120]!r}"
        assert "####" in t, f"{v}: target tanpa penanda ####"

p = build_prompt(_tok, test_rows[0]["question"])
assert not [c for c in think_content(p) if c], "prompt inference memicu thinking mode"

# Invarian yang lebih penting: teks training harus DIAWALI prompt inference.
# Kalau tidak, model dilatih pada struktur berbeda dari yang dilihat saat generate —
# degradasi diam-diam di 14 run tanpa satu pun error.
_row = train_by_variant["G1"][0]
_p, _t = build_prompt(_tok, _row["question"]), build_train_text(_tok, _row)
if not _t.startswith(_p):
    print("\nakhir PROMPT :", repr(_p[-120:]))
    print("awal  TRAIN  :", repr(_t[:len(_p)+40]))
    raise AssertionError("format train != format inference")

print(f"\nformat gate PASS")
print(f"  blok think kosong di semua varian (penanda, bukan penalaran)")
print(f"  train text diawali prompt inference — format konsisten")

# potongan instruksi/respons untuk masking loss (ChatML Qwen3)
INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART    = "<|im_start|>assistant\n"
assert INSTRUCTION_PART in t and RESPONSE_PART in t, "template bukan ChatML — sesuaikan penanda"
del _tok; gc.collect()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

====================== G1 ======================
<|im_start|>user
Solve the math problem. End your reply with '#### ' followed by the final number.

Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?<|im_end|>
<|im_start|>assistant
<think>

</think>

2 * 12 = 24
24 * 2 = 48
48 / 3 = 16
#### 16<|im_end|>


====================== G3 ======================



Let us restate what the problem gives us and identify each quantity step by step.
First, Mimi picked up 2 dozen seashells on the beach. The word "dozen" means 12,
so 2 dozen means 2 groups of 12. Therefore Mimi's shells = 2 * 12 = 24 shells.
Next, Kyle found twice as many shells as Mimi. "Twice as many" means we multiply
Mimi's amount by 2. So Kyle's shells = 24 * 2 = 48 shells. These went into his
pocket, but that does not change the count.
Finally, Leigh grabbed one-third of the

93

## Fungsi train & eval

`train_on_responses_only` memasang loss hanya pada jawaban. Tanpa ini, loss AO
didominasi token soal (72 dari 110 token) sedangkan G3 didominasi rationale —
dinamika training jadi berbeda karena alasan yang bukan granularity.

In [6]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset

def load_student(model_name, seed):
    model, tok = FastLanguageModel.from_pretrained(
        model_name, max_seq_length=MAX_SEQ, load_in_4bit=True, full_finetuning=False)
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
        use_gradient_checkpointing="unsloth", random_state=seed)
    return model, tok

def train_one(model_name, variant, seed, rows, epochs=EPOCHS, out_dir=None):
    model, tok = load_student(model_name, seed)
    ds = Dataset.from_list([{"text": build_train_text(tok, r)} for r in rows])
    trainer = SFTTrainer(
        model=model, tokenizer=tok, train_dataset=ds,
        args=SFTConfig(
            dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False,
            per_device_train_batch_size=BS, gradient_accumulation_steps=GRAD_ACC,
            num_train_epochs=epochs, learning_rate=LR, lr_scheduler_type="cosine",
            warmup_steps=8, optim="adamw_8bit", weight_decay=0.01,   # warmup_ratio deprecated di TRL v5
            fp16=not BF16, bf16=BF16,                     # lihat cell config: CC>=8, bukan is_bf16_supported()
            logging_steps=25, seed=seed, output_dir="/tmp/trainer", report_to="none"))
    trainer = train_on_responses_only(
        trainer, instruction_part=INSTRUCTION_PART, response_part=RESPONSE_PART)

    # Verifikasi masking BENAR-BENAR berlaku. Kalau penanda tidak cocok dengan
    # template, train_on_responses_only bisa no-op tanpa error -> loss ikut token
    # soal, dan AO (soal mendominasi) jadi tak sebanding dengan G3. Dicek sebelum
    # training dimulai supaya gagalnya murah.
    try:
        lab = trainer.train_dataset[0]["labels"]
        n_mask = sum(1 for x in lab if x == -100)
        assert 0 < n_mask < len(lab), \
            f"masking gagal: {n_mask}/{len(lab)} token ter-mask (harus sebagian, bukan 0/semua)"
        print(f"  loss-masking ok: {n_mask}/{len(lab)} token prompt di-mask, "
              f"{len(lab)-n_mask} token jawaban dilatih")
    except KeyError:
        print("  PERINGATAN: 'labels' tidak ada di dataset — masking TIDAK terverifikasi")

    stats = trainer.train()
    if out_dir:
        model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return model, tok, stats.training_loss

@torch.inference_mode()
def evaluate(model, tok, rows, shots=(), batch_size=16):
    try: FastLanguageModel.for_inference(model)
    except Exception: model.eval()
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    correct, out_tokens, t0 = 0, [], time.time()
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        prompts = [build_prompt(tok, r["question"], shots) for r in batch]
        enc = tok(prompts, return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_SEQ).to("cuda")
        gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        for row, seq in zip(batch, new):
            txt = tok.decode(seq, skip_special_tokens=True)
            out_tokens.append(int((seq != tok.pad_token_id).sum()))
            if extract_answer(txt) == extract_answer(row["answer"]): correct += 1
    dt = time.time() - t0
    return {"accuracy": correct/len(rows), "avg_output_tokens": float(np.mean(out_tokens)),
            "latency_s_per_item": dt/len(rows)}

COLUMNS = ["run_id","model","variant","seed","n_train","train_loss",
           "accuracy","avg_output_tokens","latency_s_per_item","wall_min"]

def append_result(row):
    """Skema dikunci. Tanpa ini, baris dgn kolom berbeda merusak CSV secara diam-diam."""
    assert set(row) == set(COLUMNS), f"skema beda: {set(row) ^ set(COLUMNS)}"
    pd.DataFrame([row])[COLUMNS].to_csv(
        RESULTS_CSV, mode="a", index=False, header=not RESULTS_CSV.exists())

def done_runs():
    return set(pd.read_csv(RESULTS_CSV).run_id) if RESULTS_CSV.exists() else set()

def free(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

---
## Smoke run

Memverifikasi rantai train → simpan → generate → parse → hitung akurasi berfungsi
utuh sebelum 14 run penuh dijalankan. Bug parsing paling sering baru terlihat di
tahap ini, dan jauh lebih murah diperbaiki di sini.

0.6B, G2, 200 sampel, 1 epoch, dievaluasi pada 50 soal.

In [7]:
# m, t, loss = train_one(MODELS[0], "G2", SEEDS[0], train_by_variant["G2"][:200], epochs=1)
# print(f"\ntraining loss: {loss:.4f}")

# # lihat output mentah sebelum percaya angka akurasi
# FastLanguageModel.for_inference(m)
# p = build_prompt(t, test_rows[0]["question"])
# g = m.generate(**t(p, return_tensors="pt").to("cuda"), max_new_tokens=MAX_NEW, do_sample=False)
# raw = t.decode(g[0][t(p, return_tensors="pt").input_ids.shape[1]:], skip_special_tokens=True)
# print(f"\n--- generasi mentah ---\n{raw}")
# print(f"\nparsed: {extract_answer(raw)} | gold: {test_rows[0]['answer']}")

# sm = evaluate(m, t, test_rows[:50])
# print("\nsmoke eval:", {k: round(v, 4) for k, v in sm.items()})
# assert sm["avg_output_tokens"] > 0, "generasi kosong"
# assert not [c for c in think_content(raw) if c], "model menghasilkan penalaran <think> saat inference"
# print("\nSMOKE PASS — pipeline utuh")
# free(m, t)

---
## Baseline pre-distillation

Mengukur akurasi kedua student **sebelum** distilasi. Tanpa angka ini, kasus di
mana distilasi justru menurunkan performa tidak akan terlihat. Biaya training: nol.

Prompt 4-shot memakai contoh G2 dengan format identik dengan target training,
sehingga perbandingannya bukan tentang perbedaan format.

In [8]:
SHOTS = train_by_variant["G2"][:4]

for name in MODELS:
    rid = f"baseline|{name.split('/')[-1]}"
    if rid in done_runs():
        print("skip", rid); continue
    mdl, tk = FastLanguageModel.from_pretrained(
        name, max_seq_length=MAX_SEQ, load_in_4bit=True, full_finetuning=False)
    res = evaluate(mdl, tk, test_rows, shots=SHOTS)
    append_result({"run_id": rid, "model": name, "variant": "pre-distill", "seed": -1,
                   "n_train": 0, "train_loss": None, **res, "wall_min": None})
    print(rid, {k: round(v, 4) for k, v in res.items()})
    free(mdl, tk)

==((====))==  Unsloth 2026.8.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

baseline|Qwen3-0.6B {'accuracy': 0.3533, 'avg_output_tokens': 87.4367, 'latency_s_per_item': 1.1203}
==((====))==  Unsloth 2026.8.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

baseline|Qwen3-1.7B {'accuracy': 0.7267, 'avg_output_tokens': 115.8967, 'latency_s_per_item': 1.7786}


---
## 14 run

| Kelompok | Jumlah |
|---|---|
| 3 granularity × 2 student × 2 seed | 12 |
| Answer-only × 2 student | 2 |

Urutan dimulai dari student 0.6B yang lebih cepat.

In [9]:
RUNS = ([(m, v, s) for m in MODELS for v in VARIANTS for s in SEEDS]
        + [(m, "AO", SEEDS[0]) for m in MODELS])
RUNS.sort(key=lambda r: MODELS.index(r[0]))        # 0.6B dulu
print(f"{len(RUNS)} run terdaftar")

for name, variant, seed in RUNS:
    rid = f"{name.split('/')[-1]}|{variant}|{seed}"
    if rid in done_runs():          # dibaca ulang tiap iterasi: aman kalau sesi mati di tengah
        print("skip", rid); continue
    print(f"\n{'='*60}\n{rid}\n{'='*60}", flush=True)
    t0 = time.time()
    rows = train_by_variant[variant]
    mdl, tk, loss = train_one(name, variant, seed, rows, out_dir=ADAPTERS/rid.replace("|", "_"))
    res = evaluate(mdl, tk, test_rows)
    row = {"run_id": rid, "model": name, "variant": variant, "seed": seed,
           "n_train": len(rows), "train_loss": loss, **res,
           "wall_min": (time.time()-t0)/60}
    append_result(row)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    free(mdl, tk)
print("\nselesai untuk sesi ini")

14 run terdaftar

Qwen3-0.6B|G1|3407
==((====))==  Unsloth 2026.8.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Unsloth 2026.8.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1455 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


...

loss-masking ok: 79/89 token prompt di-mask, 10 token jawaban dilatih


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,455 | Num Epochs = 3 | Total steps = 273
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Step,Training Loss
25,2.130038
50,0.375393
75,0.353367
100,0.354027
125,0.300449
150,0.287964
175,0.280498
200,0.217004
225,0.204948
250,0.209142


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/adapters/Qwen3-1.7B_AO_3407/tokenizer_config.json.


{'run_id': 'Qwen3-1.7B|AO|3407', 'model': 'unsloth/Qwen3-1.7B', 'variant': 'AO', 'seed': 3407, 'n_train': 1455, 'train_loss': 0.4482, 'accuracy': 0.2, 'avg_output_tokens': 5.42, 'latency_s_per_item': 0.0796, 'wall_min': 14.4596}

selesai untuk sesi ini


In [10]:
df = pd.read_csv(RESULTS_CSV)
print(f"{len(df)} / {len(RUNS)+len(MODELS)} run tercatat\n")
display(df[["run_id","variant","seed","accuracy","avg_output_tokens","latency_s_per_item"]]
        .sort_values("run_id"))

fin = df[df.variant.isin(VARIANTS)]
if len(fin):
    piv = fin.groupby(["model","variant"]).accuracy.agg(["mean","std","count"]).round(4)
    display(piv)

16 / 16 run tercatat


,run_id,variant,seed,accuracy,avg_output_tokens,latency_s_per_item
8,Qwen3-0.6B|AO|3407,AO,3407,0.123333,5.400000,0.049289
2,Qwen3-0.6B|G1|3407,G1,3407,0.430000,42.586667,0.537988
3,Qwen3-0.6B|G1|42,G1,42,0.443333,42.270000,0.462550
4,Qwen3-0.6B|G2|3407,G2,3407,0.560000,97.173333,1.106632
5,Qwen3-0.6B|G2|42,G2,42,0.573333,96.826667,1.005004
6,Qwen3-0.6B|G3|3407,G3,3407,0.560000,363.063333,2.815177
7,Qwen3-0.6B|G3|42,G3,42,0.593333,360.433333,2.790621
15,Qwen3-1.7B|AO|3407,AO,3407,0.200000,5.420000,0.079568
9,Qwen3-1.7B|G1|3407,G1,3407,0.593333,39.983333,0.598412
10,Qwen3-1.7B|G1|42,G1,42,0.580000,40.550000,0.595693


mean     std  count
model              variant                       
unsloth/Qwen3-0.6B G1       0.4367  0.0094      2
                   G2       0.5667  0.0094      2
                   G3       0.5767  0.0236      2
unsloth/Qwen3-1.7B G1       0.5867  0.0094      2
                   G2       0.7383  0.0118      2
                   G3       0.7617  0.0165      2

---

Output: `results/results.csv` (satu baris per run) dan `adapters/` (14 adapter LoRA).